# Experimenting with variations to LeNet's architecture
This is an attempt to improve the score at CIFAR-10

In [1]:
from tensorflow.keras import Sequential, layers, Model, callbacks
from tensorflow.keras.datasets import mnist, cifar10
from tensorflow import pad, expand_dims, cast, float32, int32
from tensorflow.config import list_physical_devices
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import numpy as np

# Dataset

In [4]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

y_train = cast(y_train, int32)
y_test = cast(y_test, int32)

x_train = cast(x_train, float32) / 255.0
x_test = cast(x_test, float32) / 255.0

x_val = x_train[:10000]
y_val = y_train[:10000]

# Early Stop Callback

In [6]:
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Base Implementation (to test against)

## Definition

In [8]:
class LeNet(Model):
    def __init__(self):
        super().__init__()
        self.conv1 = layers.Conv2D(filters=6, kernel_size=(5, 5), activation='tanh', input_shape=(32, 32, 3))
        self.pool2 = layers.AveragePooling2D((2, 2), strides=2)
        self.conv3 = layers.Conv2D(filters=16, kernel_size=(5, 5), activation='tanh')
        self.pool4 = layers.AveragePooling2D((2, 2), strides=2)
        self.conv5 = layers.Conv2D(filters=120, kernel_size=(5, 5), activation='tanh')
        self.flat6 = layers.Flatten()
        self.dense7 = layers.Dense(units=84, activation='tanh')
        self.dense8 = layers.Dense(units=10, activation='softmax')

    def call(self, x):
        x = self.conv1(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.pool4(x)
        x = self.conv5(x)
        x = self.flat6(x)
        x = self.dense7(x)
        x = self.dense8(x)

        return x

In [9]:
base = LeNet()
base.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
base.summary()

Model: "le_net_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ ?                      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ ?                      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Training

In [10]:
history = base.fit(
    x_train, y_train,
    epochs=100,
    callbacks=[early_stopping]
)

Epoch 1/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - accuracy: 0.3628 - loss: 1.7991
Epoch 2/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - accuracy: 0.4444 - loss: 1.5659
Epoch 3/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - accuracy: 0.4902 - loss: 1.4331
Epoch 4/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - accuracy: 0.5197 - loss: 1.3492
Epoch 5/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 24s 16ms/step - accuracy: 0.5435 - loss: 1.2803
Epoch 6/100
 296/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.5655 - loss: 1.2147

KeyboardInterrupt: 

# Test 1: Replacing tanh with ReLU